In [1]:
# ===== Imports =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ===== Load Data =====
df = pd.read_csv("../data/clustering_dataset.csv")

# ===== Scaling =====
scaler = StandardScaler()
X = scaler.fit_transform(df.values)

n = X.shape[0]

min_samples = 5
min_cluster_size = 15

In [2]:
# ===== Step A: Core Distance Computation =====

def compute_core_distances(X, k, chunk_size=500):
    n = X.shape[0]
    core_dists = np.zeros(n)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        chunk = X[start:end]
        distances = np.linalg.norm(chunk[:, None, :] - X[None, :, :], axis=2)

        sorted_dists = np.sort(distances, axis=1)
        core_dists[start:end] = sorted_dists[:, k]

    return core_dists

start_time = time.time()
core_dists = compute_core_distances(X, min_samples)

In [3]:
# ===== Step B: Mutual Reachability Distance =====

def compute_mrd_matrix(X, core_dists):
    pairwise = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)

    mrd = np.maximum(pairwise, core_dists[:, None])
    mrd = np.maximum(mrd, core_dists[None, :])

    return mrd

mrd_matrix = compute_mrd_matrix(X, core_dists)

In [4]:
# ===== Step C: Minimum Spanning Tree (Prim's Algorithm) =====

def prim_mst(mrd):
    n = mrd.shape[0]
    selected = np.zeros(n, dtype=bool)
    selected[0] = True

    edges = []

    for _ in range(n - 1):
        min_edge = (None, None, np.inf)

        for i in range(n):
            if selected[i]:
                for j in range(n):
                    if not selected[j] and mrd[i, j] < min_edge[2]:
                        min_edge = (i, j, mrd[i, j])

        i, j, w = min_edge
        edges.append((i, j, w))
        selected[j] = True

    return edges

mst_edges = prim_mst(mrd_matrix)

KeyboardInterrupt: 

In [ ]:
# ===== Step E/F: Condensed Tree & Stability =====

uf = UnionFind(n)
clusters = {}
birth_lambda = {}

for i in range(n):
    clusters[i] = {i}
    birth_lambda[i] = 0

for i, j, weight in mst_edges:
    lambda_val = 1.0 / weight if weight > 0 else 0
    ri, rj = uf.find(i), uf.find(j)

    if ri != rj:
        size_i = len(clusters[ri])
        size_j = len(clusters[rj])

        uf.union(ri, rj)
        new_root = uf.find(ri)

        clusters[new_root] = clusters[ri].union(clusters[rj])
        birth_lambda[new_root] = lambda_val

labels = -1 * np.ones(n)
cluster_id = 0

for root in clusters:
    if len(clusters[root]) >= min_cluster_size:
        for idx in clusters[root]:
            labels[idx] = cluster_id
        cluster_id += 1

run_time = time.time() - start_time

In [ ]:
# ===== Visualization =====

plt.figure(figsize=(6,6))
unique_labels = set(labels)

for label in unique_labels:
    mask = labels == label
    if label == -1:
        plt.scatter(X[mask,0], X[mask,1], c='black', s=10, label="Noise")
    else:
        plt.scatter(X[mask,0], X[mask,1], s=10)

plt.title("HDBSCAN Clusters")
plt.show()

# ===== Metrics (exclude noise) =====

mask = labels != -1

if len(set(labels[mask])) > 1:
    sil = silhouette_score(X[mask], labels[mask])
    db = davies_bouldin_score(X[mask], labels[mask])
    ch = calinski_harabasz_score(X[mask], labels[mask])
else:
    sil, db, ch = "", "", ""

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = np.sum(labels == -1)

print("Clusters found:", n_clusters)
print("Noise points:", n_noise)

# ===== Logging =====

results_path = "../results.csv"

row = [
    "hdbscan",
    n_clusters,
    "",
    min_samples,
    "",
    sil,
    db,
    ch,
    n_noise,
    run_time
]

header = "model,n_clusters,epsilon,min_pts,inertia,silhouette,davies_bouldin_score,calinski_harabasz_score,n_noise,run_time\n"

if not os.path.exists(results_path):
    with open(results_path, "w") as f:
        f.write(header)

with open(results_path, "a") as f:
    f.write(",".join(map(str, row)) + "\n")